# 05 — Exploration des tables gold et choix des graphiques

Ce notebook explore les cinq tables gold (PostgreSQL) pour répondre aux questions de la directrice des
opérations, et choisit, graphique par graphique, la forme qui sera reprise dans le dashboard. Il se termine par un
**avant / après** : le camembert des causes de retard refait selon les principes de perception.

**pandas uniquement sur les tables gold** : elles comptent de 168 à 6 805 lignes. Les 7 millions de vols restent
dans le data lake, agrégés par Spark dans `03_gold`.

**Règles appliquées à chaque graphique (séance S04)**

| Règle | Pourquoi |
|---|---|
| Le titre énonce la **conclusion**, le sous-titre dit ce qui est mesuré | le lecteur pressé lit le titre et s'arrête là |
| Unités sur les axes, barres qui partent de 0 | une barre code par sa longueur : tronquer l'axe exagère les écarts |
| Pas de camembert, pas d'arc-en-ciel | on compare mal les angles et les aires, et des couleurs sans ordre ne portent aucun sens |
| **Une seule couleur d'accent**, et seulement si elle a un sens | ici le bleu veut toujours dire « là où agir » ; le reste est gris |
| Étiquettes directes plutôt que légende | pas d'aller-retour de l'œil entre la légende et les marques |

Les titres sont **calculés à partir des données** : relancé sur d'autres données, le notebook garde des titres
vrais.

## 1. Lecture des tables et style commun

**Ré-agréger à partir des comptes.** Dès qu'on regroupe plusieurs lignes (plusieurs mois, plusieurs compagnies),
on recalcule les taux à partir des comptes (`somme des vols à l'heure / somme des vols opérés`), jamais en
faisant la moyenne des pourcentages : un mois de 120 000 vols pèse plus qu'un mois de 6 000. Les tables de
`03_gold` contiennent ces comptes (`vols_operes`, `vols_a_l_heure`, `vols_annules`) ; si une table ne les a
pas (contrat strict), `avec_comptes()` les reconstitue à partir des pourcentages.

In [ ]:
import sys
sys.path.append("/home/jovyan/work")
from commun.donnees import lire

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

# Palette : une seule couleur d'accent, qui veut toujours dire « là où agir » ; le reste en gris
ACCENT = "#257de0"
GRIS = "#b4b4b4"
ENCRE, ENCRE_2, GRILLE, AXE = "#191919", "#666666", "#e7e7e7", "#c2c2c2"
SEQUENTIEL = LinearSegmentedColormap.from_list(   # une seule teinte, du clair (peu) au foncé (beaucoup)
    "bleus", ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0e3d7e"])

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXE, "axes.labelcolor": ENCRE_2, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": ENCRE, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRILLE, "grid.linewidth": 0.8,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2, "text.color": ENCRE, "font.size": 10,
})
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre",
        "novembre", "décembre"]
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
EN_PCT = FuncFormatter(lambda v, _: f"{v:.0f} %")


def pct(x, d=1):
    """74.83 → « 74,8 % » (virgule décimale)."""
    return f"{x:.{d}f}".replace(".", ",") + " %"


def milliers(n):
    return f"{int(round(n)):,}".replace(",", " ")


def titrer(ax, conclusion, mesure):
    """Titre = la conclusion ; sous-titre gris = ce qui est mesuré."""
    ax.set_title(conclusion, pad=26)
    ax.text(0, 1.02, mesure, transform=ax.transAxes, color=ENCRE_2, fontsize=9.5, va="bottom")


def avec_comptes(df, vols="vols"):
    """Ajoute vols_operes / vols_a_l_heure / vols_annules s'ils manquent (approximation : les détournés,
    ≈ 0,25 % des vols, restent alors comptés comme opérés)."""
    df = df.copy()
    if "vols_operes" not in df:
        df["vols_annules"] = df[vols] * df["pct_annules"] / 100
        df["vols_operes"] = df[vols] - df["vols_annules"]
        df["vols_a_l_heure"] = df["vols_operes"] * df["pct_a_l_heure"] / 100
    return df


ponctualite = avec_comptes(lire("gold_ponctualite_mensuelle"))
causes = lire("gold_causes_retard")
aeroports = avec_comptes(lire("gold_aeroports"), vols="departs")
heure_jour = lire("gold_heure_jour")
routes = lire("gold_routes")
ponctualite["mois"] = pd.to_datetime(ponctualite["mois"])

for nom, table in [("gold_ponctualite_mensuelle", ponctualite), ("gold_causes_retard", causes),
                   ("gold_aeroports", aeroports), ("gold_heure_jour", heure_jour), ("gold_routes", routes)]:
    print(f"{nom:<27} {len(table):>5} lignes")

NATIONAL = 100 * ponctualite.vols_a_l_heure.sum() / ponctualite.vols_operes.sum()
print(f"\nPonctualité nationale 2024 : {pct(NATIONAL)} des vols opérés arrivés à l'heure "
      f"({milliers(ponctualite.vols.sum())} vols programmés)")